# Org

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import warnings
import glob
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")

pandas version: 2.0.3
numpy version: 1.24.4


In [ ]:
BASE_DIR = Path("/../coloc_500kb_definedloci")
OUT_DIR  = Path("/../coloc_500kb_definedloci/results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def load_all(pattern, label):
    files = glob.glob(str(BASE_DIR / f"**/{pattern}"), recursive=True)
    if not files:
        print(f"No files found for pattern: {pattern}")
        return pd.DataFrame()
    print(f"Found {len(files)} {label} files")
    dfs = []
    for f in files:
        tissue = Path(f).parent.name
        df = pd.read_csv(f, sep="\t")
        df["tissue_dir"] = tissue
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

def normalize_chr_prefix(val):
    if pd.isna(val):
        return val
    s = str(val)
    if s.startswith("CHR") or s.startswith("Chr"):
        s = "chr" + s[3:]
    return s

def normalize_all_chr_columns(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map(normalize_chr_prefix)
    return df

def natural_locus_rank(locus):
    # expected: chr<chrom>:<start>-<end>
    m = re.match(r"^chr([0-9]+|X|Y|MT):(\d+)-(\d+)$", str(locus))
    if not m:
        return (999, 0)
    chrom = m.group(1)
    start = int(m.group(2))
    if chrom.isdigit():
        rank = int(chrom)
    else:
        rank = {"X": 23, "Y": 24, "MT": 25}.get(chrom, 999)
    return (rank, start)

def parse_variant_pos(v):
    """
    Accepts:
      - GWAS-like: chr1:12345:A:G  -> ('chr1', 12345)
      - GTEx-like: chr1_12345_A_G_b38 -> ('chr1', 12345)
    Returns (chr, pos) or (None, None).
    """
    if pd.isna(v):
        return (None, None)
    s = str(v)

    # GWAS style
    m = re.match(r"^(chr[0-9XYMT]+):(\d+):", s)
    if m:
        return (m.group(1), int(m.group(2)))

    # GTEx style
    m = re.match(r"^(chr[0-9XYMT]+)_(\d+)_", s)
    if m:
        return (m.group(1), int(m.group(2)))

    return (None, None)

# ---------------------------------------------------------------------
# Load inputs

female = load_all("coloc_female_full.tsv", "female")
male   = load_all("coloc_male_full.tsv",   "male")

# ---------------------------------------------------------------------
# Keep key columns + tolerate missing

pp_cols = ["PP.H0","PP.H1","PP.H2","PP.H3","PP.H4"]
extra_cols = [
    "top_gwas_variant","top_gwas_pval",
    "lead_eqtl_id","lead_eqtl_p",
    "n_snps_overlap",  # keep only this count
    "locus_id","tissue","tissue_dir","gene_id","gene_name"
]

def prepare(df):
    if df.empty:
        return df
    df = df.copy()
    for c in pp_cols + extra_cols:
        if c not in df.columns:
            df[c] = np.nan
    if "tissue" not in df.columns or df["tissue"].isna().all():
        df["tissue"] = df["tissue_dir"]
    # normalize IDs
    df["locus_id"] = df["locus_id"].map(normalize_chr_prefix)
    df = normalize_all_chr_columns(df, ["top_gwas_variant","lead_eqtl_id"])
    return df

female = prepare(female)
male   = prepare(male)

# ---------------------------------------------------------------------
# Add top-in-locus variants for GWAS & eQTL (per sex)
# (We still compute "in_gene" internally, but we'll DROP them later.)
# ---------------------------------------------------------------------
def add_top_variants(df, sex_label):
    if df.empty:
        cols = [
            f"{sex_label}_{k.replace('.','_')}" for k in pp_cols
        ] + [
            "gene_name","gene_id","locus_id","tissue",
            f"{sex_label}_n_snps_overlap",
            # in-gene (will be dropped later)
            f"{sex_label}_top_gwas_variant_in_gene",
            f"{sex_label}_top_gwas_pval_in_gene",
            f"{sex_label}_top_eQTL_variant_in_gene",
            f"{sex_label}_top_eQTL_pval_in_gene",
            # in-locus (kept)
            f"{sex_label}_top_gwas_variant_in_locus",
            f"{sex_label}_top_gwas_pval_in_locus",
            f"{sex_label}_top_eQTL_variant_in_locus",
            f"{sex_label}_top_eQTL_pval_in_locus",
        ]
        return pd.DataFrame(columns=cols)

    # --- top GWAS in gene
    gw_gene = (df.sort_values("top_gwas_pval", na_position="last")
                 .groupby(["gene_name","gene_id","locus_id","tissue"], as_index=False)
                 .first()[["gene_name","gene_id","locus_id","tissue","top_gwas_variant","top_gwas_pval"]])
    gw_gene = gw_gene.rename(columns={
        "top_gwas_variant": f"{sex_label}_top_gwas_variant_in_gene",
        "top_gwas_pval":    f"{sex_label}_top_gwas_pval_in_gene"
    })

    # --- top GWAS in locus
    gw_locus = (df.sort_values("top_gwas_pval", na_position="last")
                  .groupby(["locus_id","tissue"], as_index=False)
                  .first()[["locus_id","tissue","top_gwas_variant","top_gwas_pval"]])
    gw_locus = gw_locus.rename(columns={
        "top_gwas_variant": f"{sex_label}_top_gwas_variant_in_locus",
        "top_gwas_pval":    f"{sex_label}_top_gwas_pval_in_locus"
    })

    # --- top eQTL in gene
    eq_gene = (df.sort_values("lead_eqtl_p", na_position="last")
                 .groupby(["gene_name","gene_id","locus_id","tissue"], as_index=False)
                 .first()[["gene_name","gene_id","locus_id","tissue","lead_eqtl_id","lead_eqtl_p"]])
    eq_gene = eq_gene.rename(columns={
        "lead_eqtl_id": f"{sex_label}_top_eQTL_variant_in_gene",
        "lead_eqtl_p":  f"{sex_label}_top_eQTL_pval_in_gene"
    })

    # --- top eQTL in locus
    eq_locus = (df.sort_values("lead_eqtl_p", na_position="last")
                  .groupby(["locus_id","tissue"], as_index=False)
                  .first()[["locus_id","tissue","lead_eqtl_id","lead_eqtl_p"]])
    eq_locus = eq_locus.rename(columns={
        "lead_eqtl_id": f"{sex_label}_top_eQTL_variant_in_locus",
        "lead_eqtl_p":  f"{sex_label}_top_eQTL_pval_in_locus"
    })

    # --- PP.* and n_snps_overlap (max)
    agg = (df.groupby(["gene_name","gene_id","locus_id","tissue"], as_index=False)
             .agg(**{f"{sex_label}_{k.replace('.','_')}": (k, "max") for k in pp_cols},
                  **{f"{sex_label}_n_snps_overlap": ("n_snps_overlap", "max")})
          )

    out = (agg.merge(gw_gene, on=["gene_name","gene_id","locus_id","tissue"], how="left")
              .merge(gw_locus, on=["locus_id","tissue"], how="left")
              .merge(eq_gene, on=["gene_name","gene_id","locus_id","tissue"], how="left")
              .merge(eq_locus, on=["locus_id","tissue"], how="left"))

    # Normalize any lingering CHR→chr in added columns
    out = normalize_all_chr_columns(out, [
        f"{sex_label}_top_gwas_variant_in_gene",
        f"{sex_label}_top_gwas_variant_in_locus",
        f"{sex_label}_top_eQTL_variant_in_gene",
        f"{sex_label}_top_eQTL_variant_in_locus",
    ])
    return out

female_summary = add_top_variants(female, "Female")
male_summary   = add_top_variants(male,   "Male")

# ---------------------------------------------------------------------
# Merge Female + Male summaries

merged = pd.merge(
    female_summary, male_summary,
    on=["gene_name","gene_id","locus_id","tissue"], how="outer"
)

# Fill PPs for stable interpretation
for sex in ["Female","Male"]:
    for k in ["PP_H0","PP_H1","PP_H2","PP_H3","PP_H4"]:
        col = f"{sex}_{k}"
        if col in merged.columns:
            merged[col] = merged[col].fillna(0.0)

# Integer-only overlaps
for col in ["Female_n_snps_overlap","Male_n_snps_overlap"]:
    if col in merged.columns:
        merged[col] = merged[col].fillna(0).astype("Int64")

# Unified “best” eQTL per locus (by p-value across sexes)
def best_eqtl_locus(row):
    f_id, f_p = row.get("Female_top_eQTL_variant_in_locus"), row.get("Female_top_eQTL_pval_in_locus")
    m_id, m_p = row.get("Male_top_eQTL_variant_in_locus"),   row.get("Male_top_eQTL_pval_in_locus")
    if pd.notna(f_p) and (pd.isna(m_p) or f_p <= m_p):
        return pd.Series({"top_eQTL_variant_in_locus": f_id, "top_eQTL_pval_in_locus": f_p})
    if pd.notna(m_p):
        return pd.Series({"top_eQTL_variant_in_locus": m_id, "top_eQTL_pval_in_locus": m_p})
    return pd.Series({"top_eQTL_variant_in_locus": np.nan, "top_eQTL_pval_in_locus": np.nan})

merged = pd.concat([merged, merged.apply(best_eqtl_locus, axis=1)], axis=1)
merged[["top_eQTL_variant_in_locus"]] = merged[["top_eQTL_variant_in_locus"]].applymap(normalize_chr_prefix)

# ---------------------------------------------------------------------
# Compare unified top eQTL (in locus) to each sex's top GWAS (in locus)
#  - "match" if same chr & pos
#  - "diff_chr" if chromosomes differ
#  - otherwise "<N>bp" absolute distance

def compare_eqtl_to_gwas(eqtl_var, gwas_var):
    eq_chr, eq_pos = parse_variant_pos(eqtl_var)
    gw_chr, gw_pos = parse_variant_pos(gwas_var)
    if eq_chr is None or gw_chr is None:
        return np.nan
    if eq_chr != gw_chr:
        return "diff_chr"
    if eq_pos == gw_pos:
        return "match"
    return f"{abs(eq_pos - gw_pos)}bp"

merged["eQTL_vs_FemaleGWAS"] = merged.apply(
    lambda r: compare_eqtl_to_gwas(r.get("top_eQTL_variant_in_locus"), r.get("Female_top_gwas_variant_in_locus")),
    axis=1
)
merged["eQTL_vs_MaleGWAS"] = merged.apply(
    lambda r: compare_eqtl_to_gwas(r.get("top_eQTL_variant_in_locus"), r.get("Male_top_gwas_variant_in_locus")),
    axis=1
)

# ---------------------------------------------------------------------
# Category & differences 

H4_STRONG = 0.8
def classify(row):
    f, m = row["Female_PP_H4"], row["Male_PP_H4"]
    if f >= H4_STRONG and m >= H4_STRONG:
        return "Strong in Both"
    elif m >= H4_STRONG and f < H4_STRONG:
        return "Strong in Males only"
    elif f >= H4_STRONG and m < H4_STRONG:
        return "Strong in Females only"
    else:
        return "Weak / None"

merged["Coloc_Category"] = merged.apply(classify, axis=1)
merged["Abs_PP_H4_diff"] = (merged["Female_PP_H4"] - merged["Male_PP_H4"]).abs()

# Ensure locus_id normalized (again)
merged["locus_id"] = merged["locus_id"].map(normalize_chr_prefix)
merged["_rank"] = merged["locus_id"].map(natural_locus_rank)

# ---------------------------------------------------------------------
# Female/Male interpretation strings (based on max PP.H*)

H4_STRONG = 0.80

def _interpret(h0, h1, h2, h3, h4):
    vals = {"PP.H0": float(h0), "PP.H1": float(h1), "PP.H2": float(h2),
            "PP.H3": float(h3), "PP.H4": float(h4)}
    top = max(vals, key=vals.get)
    v = vals[top]

    if top == "PP.H4":
        if v >= H4_STRONG:
            label = "Strong colocalization"
        else:
            label = "Suggestive colocalization"
    elif top == "PP.H3":
        label = "Distinct signals, different variants"
    elif top == "PP.H2":
        label = "eQTL-only association"
    elif top == "PP.H1":
        label = "GWAS-only association"
    else:
        label = "No signal"

    return f"{label} ({top}={v:.2f})"

merged["Female_Interpretation"] = merged.apply(
    lambda r: _interpret(r["Female_PP_H0"], r["Female_PP_H1"], r["Female_PP_H2"], r["Female_PP_H3"], r["Female_PP_H4"]),
    axis=1
)
merged["Male_Interpretation"] = merged.apply(
    lambda r: _interpret(r["Male_PP_H0"], r["Male_PP_H1"], r["Male_PP_H2"], r["Male_PP_H3"], r["Male_PP_H4"]),
    axis=1
)

# ---------------------------------------------------------------------
# DROP all *_in_gene columns
drop_cols = [
    "Female_top_gwas_variant_in_gene","Female_top_gwas_pval_in_gene",
    "Female_top_eQTL_variant_in_gene","Female_top_eQTL_pval_in_gene",
    "Male_top_gwas_variant_in_gene","Male_top_gwas_pval_in_gene",
    "Male_top_eQTL_variant_in_gene","Male_top_eQTL_pval_in_gene",
]
for c in drop_cols:
    if c in merged.columns:
        merged = merged.drop(columns=c)

# ---------------------------------------------------------------------
# Final columns
final_cols = [
    # locus & IDs
    "locus_id","gene_name","gene_id","tissue",

    # GWAS tops per sex (IN LOCUS)
    "Female_top_gwas_variant_in_locus","Female_top_gwas_pval_in_locus",
    "Male_top_gwas_variant_in_locus","Male_top_gwas_pval_in_locus",

    # Unified best eQTL (IN LOCUS)
    "top_eQTL_variant_in_locus","top_eQTL_pval_in_locus",

    # NEW comparisons
    "eQTL_vs_FemaleGWAS","eQTL_vs_MaleGWAS",

    # coloc PP + counts
    "Female_PP_H0","Female_PP_H1","Female_PP_H2","Female_PP_H3","Female_PP_H4",
    "Male_PP_H0","Male_PP_H1","Male_PP_H2","Male_PP_H3","Male_PP_H4",

    # NEW: human-readable interpretations
    "Female_Interpretation","Male_Interpretation",

    # counts
    "Female_n_snps_overlap","Male_n_snps_overlap",

    # diffs & category
    "Abs_PP_H4_diff","Coloc_Category",
]
final_cols = [c for c in final_cols if c in merged.columns]

# Sort: locus (chr-natural), then gene, then tissue
final = (merged[final_cols + ["_rank"]]
         .sort_values(by=["_rank","gene_name","tissue"], kind="mergesort")
         .drop(columns=["_rank"]))

# Ensure integers remain integers on disk
int_cols = ["Female_n_snps_overlap","Male_n_snps_overlap"]
for c in int_cols:
    if c in final.columns:
        final[c] = final[c].astype("Int64")

## Save
out_path = OUT_DIR / "sex_specific_coloc_PP_interpreted.tsv"
final.to_csv(out_path, sep="\t", index=False)
print(f"Combined and saved: {out_path}")

# Check MHC
chr6:28,510,120-33,480,577 hg38

In [ ]:
import pandas as pd
import re

# Load the final merged colocalization results
results_file = "/../coloc_500kb_definedloci/results/sex_specific_coloc_PP_interpreted.tsv"

print("Loading colocalization results...")
df = pd.read_csv(results_file, sep='\t')

print(f"Total rows in results: {len(df):,}")
print(f"Unique loci: {df['locus_id'].nunique():,}")

# Define MHC region (GRCh38)
MHC_START = 28000000
MHC_END = 34000000

# Function to check if a locus overlaps MHC
def is_mhc_locus(locus_id):
    if pd.isna(locus_id):
        return False
    # Parse locus_id format: chr6:start-end
    match = re.match(r'chr6:(\d+)-(\d+)', str(locus_id))
    if not match:
        return False
    start = int(match.group(1))
    end = int(match.group(2))
    # Check for overlap
    return (start <= MHC_END) and (end >= MHC_START)

# Filter for MHC
df['is_mhc'] = df['locus_id'].apply(is_mhc_locus)
mhc_results = df[df['is_mhc']].copy()

print(f"\n{'='*60}")
print(f"MHC REGION CHECK (chr6:{MHC_START:,}-{MHC_END:,})")
print(f"{'='*60}")

if len(mhc_results) > 0:
    print(f"\nFound {len(mhc_results)} MHC colocalization results")
    print(f"  Unique MHC loci: {mhc_results['locus_id'].nunique()}")
    print(f"  Unique genes: {mhc_results['gene_name'].nunique()}")
    print(f"  Tissues represented: {mhc_results['tissue'].nunique()}")
    
    print("\n--- MHC Loci Found ---")
    for locus in mhc_results['locus_id'].unique():
        count = len(mhc_results[mhc_results['locus_id'] == locus])
        print(f"  {locus}: {count} gene-tissue pairs")
    
    print("\n--- Strong Colocalization in MHC (PP.H4 >= 0.80) ---")
    strong_mhc = mhc_results[
        (mhc_results['Coloc_Category'].isin(['Strong in Both', 'Strong in Males only', 'Strong in Females only']))
    ]
    if len(strong_mhc) > 0:
        print(f"  Found {len(strong_mhc)} strong MHC colocalizations:")
        for _, row in strong_mhc.iterrows():
            print(f"    {row['locus_id']} | {row['gene_name']} | {row['tissue']} | {row['Coloc_Category']}")
            print(f"      Female PP.H4={row['Female_PP_H4']:.3f}, Male PP.H4={row['Male_PP_H4']:.3f}")
    else:
        print("  No strong colocalizations found in MHC")
    
    # Save MHC results
    #mhc_output = "coloc-GTEx-cisQTL-v10/MHC_coloc_results.tsv"
    #mhc_results.to_csv(mhc_output, sep='\t', index=False)
    #print(f"\n MHC results saved to: {mhc_output}")
    
else:
    print("\nNo MHC region results found in colocalization output")

# Also check all chr6 loci for context
chr6_results = df[df['locus_id'].str.startswith('chr6:', na=False)]
print(f"\n--- All Chromosome 6 Loci ---")
print(f"Total chr6 results: {len(chr6_results)}")
if len(chr6_results) > 0:
    print("Chr6 loci:")
    for locus in sorted(chr6_results['locus_id'].unique()):
        count = len(chr6_results[chr6_results['locus_id'] == locus])
        is_mhc_flag = " [MHC]" if is_mhc_locus(locus) else ""
        print(f"  {locus}{is_mhc_flag}: {count} results")